# 🟦 سلول ۱ — بررسی GPU و نصب کتابخانه‌ها


In [6]:
#@title 1. بررسی GPU و نصب کتابخانه‌ها
!nvidia-smi

!pip install -q transformers datasets accelerate sentencepiece evaluate
!pip install -q --upgrade torch

import torch
print(f"\n✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
print(f"✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Mon Nov  2 01:03:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# 🟦 سلول ۲ — اتصال به Google Drive و ساخت ساختار پروژه pars_corrector


In [1]:
#@title 2. اتصال به Google Drive و ساخت ساختار پروژه
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_NAME = "pars_corrector"
PROJECT_DIR = f"/content/drive/MyDrive/{PROJECT_NAME}"
DATA_DIR = f"{PROJECT_DIR}/data"
MODEL_DIR = f"{PROJECT_DIR}/models"
CHECKPOINT_DIR = f"{PROJECT_DIR}/checkpoints"
OUTPUT_DIR = f"{PROJECT_DIR}/output"
NOTEBOOK_DIR = f"{PROJECT_DIR}/notebooks"

for d in [PROJECT_DIR, DATA_DIR, MODEL_DIR, CHECKPOINT_DIR, OUTPUT_DIR, NOTEBOOK_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f"✅ {d}")

print(f"\n📁 مسیر پروژه: {PROJECT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ /content/drive/MyDrive/pars_corrector
✅ /content/drive/MyDrive/pars_corrector/data
✅ /content/drive/MyDrive/pars_corrector/models
✅ /content/drive/MyDrive/pars_corrector/checkpoints
✅ /content/drive/MyDrive/pars_corrector/output
✅ /content/drive/MyDrive/pars_corrector/notebooks

📁 مسیر پروژه: /content/drive/MyDrive/pars_corrector


# 🟦 سلول ۳ — دانلود دیتاست تصحیح غلط املایی فارسی و ذخیره در Drive


In [2]:
#@title 3. دانلود و ذخیره دیتاست در Drive
from datasets import load_dataset

# ---------- گزینه اول (پیشنهادی) ----------
DATASET_NAME = "aictsharif/perisan_texts_with_typos"
LOCAL_DATASET_PATH = f"{DATA_DIR}/persian_typo_dataset"

try:
    dataset = load_dataset(DATASET_NAME)
    print("✅ دیتاست از HuggingFace بارگذاری شد")
    print(f"📊 ساختار: {dataset}")
    print(f"📊 Train size: {len(dataset['train'])}")
    if 'validation' in dataset:
        print(f"📊 Validation size: {len(dataset['validation'])}")

    dataset.save_to_disk(LOCAL_DATASET_PATH)
    print(f"\n✅ دیتاست در Drive ذخیره شد: {LOCAL_DATASET_PATH}")

except Exception as e:
    print(f"⚠️ خطا در بارگذاری: {e}")
    print("→ گزینه دوم را امتحان کن یا از سلول 3b (CSV) استفاده کن.")

# ---------- گزینه دوم (در صورت نیاز uncomment کن) ----------
# DATASET_NAME = "persian_news_typos"
# LOCAL_DATASET_PATH = f"{DATA_DIR}/persian_news_typos"
# dataset = load_dataset(DATASET_NAME)
# dataset.save_to_disk(LOCAL_DATASET_PATH)

✅ دیتاست از HuggingFace بارگذاری شد
📊 ساختار: DatasetDict({
    train: Dataset({
        features: ['with_typos', 'fixed'],
        num_rows: 22458
    })
    test: Dataset({
        features: ['with_typos', 'fixed'],
        num_rows: 5615
    })
})
📊 Train size: 22458


Saving the dataset (0/1 shards):   0%|          | 0/22458 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5615 [00:00<?, ? examples/s]


✅ دیتاست در Drive ذخیره شد: /content/drive/MyDrive/pars_corrector/data/persian_typo_dataset


# 🟦 سلول ۳b — (جایگزین) بارگذاری دیتاست سفارشی از CSV در Drive


In [4]:
#@title 3b. بارگذاری دیتاست سفارشی از CSV (اختیاری)
import pandas as pd
from datasets import Dataset, DatasetDict

CSV_PATH = f"{DATA_DIR}/my_typo_dataset.csv"   # مسیر فایل خودت را بگذار

def load_custom_csv(csv_path):
    df = pd.read_csv(csv_path)
    print(f"📊 ستون‌ها: {df.columns.tolist()}")
    print(f"📊 تعداد ردیف: {len(df)}")
    print(df.head())

    # فرض: ستون اول = متن دارای غلط، ستون دوم = متن صحیح
    col_input = df.columns[0]
    col_target = df.columns[1]
    df = df.rename(columns={col_input: "input", col_target: "target"})
    df = df[["input", "target"]].dropna()

    ds = Dataset.from_pandas(df)
    ds = ds.train_test_split(test_size=0.1, seed=42)
    return DatasetDict({"train": ds["train"], "validation": ds["test"]})

# dataset = load_custom_csv(CSV_PATH)   # ← در صورت استفاده uncomment کن

# حذف کتابخانه غیر ضروری

In [4]:
#@title ✂️ حذف torchvision برای جلوگیری از خطا
!pip uninstall -y torchvision torchaudio

Found existing installation: torchvision 0.26.0+cu130
Uninstalling torchvision-0.26.0+cu130:
  Successfully uninstalled torchvision-0.26.0+cu130
Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130


# ✅ سلول ۴ کامل و نهایی (کپی کن و جایگزین سلول ۴ فعلی کن)


In [4]:
#@title 4. بارگذاری mT5 با کش در Drive
# ---------- تشخیص و نصب خودکار در صورت نیاز ----------
import importlib, subprocess, sys

def ensure(pkg, name=None):
    name = name or pkg
    try:
        importlib.import_module(name)
        print(f"✅ {pkg} موجود است")
    except ImportError:
        print(f"⬇️ نصب {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"✅ {pkg} نصب شد")

ensure("transformers")
ensure("sentencepiece")
ensure("torch")

# ---------- import ----------
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import transformers, torch, os

print(f"\n📦 transformers: {transformers.__version__}")
print(f"📦 torch: {torch.__version__}")

# ---------- ⬇️⬇️ اینجا کش را تعریف می‌کنیم ⬇️⬇️ ----------
MODEL_NAME = "google/mt5-small"
CACHE_DIR = f"{MODEL_DIR}/hf_cache"        # ← کش در Google Drive
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"📁 پوشه کش: {CACHE_DIR}")

# ---------- ⬇️⬇️ اینجا cache_dir را به from_pretrained می‌دهیم ⬇️⬇️ ----------
try:
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        cache_dir=CACHE_DIR,               # ← این خط اضافه شد
    )
    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_NAME,
        cache_dir=CACHE_DIR,               # ← این خط اضافه شد
    )

    # خاموش‌کردن هشدار tie (اختیاری)
    model.config.tie_word_embeddings = False

    print(f"\n✅ مدل با موفقیت بارگذاری شد: {MODEL_NAME}")
    print(f"📊 پارامترها: {sum(p.numel() for p in model.parameters()):,}")
    print(f"📊 توکنایزر: {type(tokenizer).__name__}")
except Exception as e:
    print(f"\n❌ خطا در بارگذاری مدل: {e}")
    raise

✅ transformers موجود است
✅ sentencepiece موجود است
✅ torch موجود است

📦 transformers: 5.21.0
📦 torch: 2.15.0+cu132
📁 پوشه کش: /content/drive/MyDrive/pars_corrector/models/hf_cache


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.20GB            

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


✅ مدل با موفقیت بارگذاری شد: google/mt5-small
📊 پارامترها: 300,176,768
📊 توکنایزر: T5Tokenizer


# 📋 مرحله ۲ — بررسی سریع کش (اختیاری ولی توصیه می‌شود)
    در یک سلول جدید این را بزن تا مطمئن شوی کش ساخته شده

In [5]:
#@title 🩺 بررسی کش مدل
!du -sh {CACHE_DIR}
!ls -lh {CACHE_DIR}

2.3G	/content/drive/MyDrive/pars_corrector/models/hf_cache
total 8.5K
drwx------ 4 root root 4.0K Nov  2 01:13 blobs
-rw------- 1 root root  191 Nov  2 01:13 CACHEDIR.TAG
drwx------ 6 root root 4.0K Nov  2 01:13 models--google--mt5-small


# 📋 مرحله ۳ — تأیید وجود دیتاست در حافظه
قبل از سلول ۵، مطمئن شو dataset در حافظه هست:



In [6]:
#@title 🩺 بررسی دیتاست
try:
    print(f"📊 ستون‌ها: {dataset['train'].column_names}")
    print(f"📊 تعداد نمونه train: {len(dataset['train'])}")
    print(f"📊 تعداد نمونه validation: {len(dataset['validation']) if 'validation' in dataset else 'N/A'}")
    print(f"\n📝 نمونه اول:")
    print(dataset['train'][0])
except NameError:
    print("❌ dataset تعریف نشده → سلول ۳ (یا ۳b) را دوباره اجرا کن")

📊 ستون‌ها: ['with_typos', 'fixed']
📊 تعداد نمونه train: 22458
📊 تعداد نمونه validation: N/A

📝 نمونه اول:
{'with_typos': 'وزیر امور خارجه در سخنانی اظعار داشت که ایران هقگز اجازه دخالت خارجی را نخواهد داد.', 'fixed': 'وزیر امور خارجه در سخنانی اظهار داشت که ایران هرگز اجازه دخالت خارجی را نخواهد داد.'}


# ✅ راه‌حل: سلول ۵ جدید (جایگزین سلول ۵ فعلی)
این سلول هم نام ستون‌ها را اصلاح می‌کند، هم validation می‌سازد، هم توکنایز می‌کند — همه در یک سلول:



In [7]:
#@title 5. نرمال‌سازی دیتاست + توکنایز (نسخه اصلاح‌شده)
from datasets import DatasetDict
import numpy as np

# ---------- گام ۱: اصلاح نام ستون‌ها ----------
print("🔧 گام ۱: نرمال‌سازی نام ستون‌ها...")
current_cols = dataset["train"].column_names
print(f"   ستون‌های فعلی: {current_cols}")

if "with_typos" in current_cols and "fixed" in current_cols:
    dataset = dataset.rename_columns({
        "with_typos": "input",
        "fixed": "target",
    })
    print("   ✅ with_typos → input, fixed → target")
elif "input" in current_cols and "target" in current_cols:
    print("   ✅ ستون‌ها از قبل درست‌اند")
else:
    raise ValueError(f"❌ ستون‌های ناشناخته: {current_cols}")

# ---------- گام ۲: ساخت validation split ----------
print("\n🔧 گام ۲: بررسی تقسیم train/validation...")
if "validation" not in dataset:
    print("   ⚠️ validation وجود ندارد → تقسیم ۹۰/۱۰ انجام می‌شود")
    split = dataset["train"].train_test_split(test_size=0.1, seed=42)
    dataset = DatasetDict({
        "train": split["train"],
        "validation": split["test"],
    })
    print(f"   ✅ train: {len(dataset['train'])}")
    print(f"   ✅ validation: {len(dataset['validation'])}")
else:
    print(f"   ✅ validation از قبل موجود: {len(dataset['validation'])}")

# ---------- گام ۳: ذخیره نسخه نرمال‌شده در Drive ----------
from datasets import load_from_disk
normalized_path = f"{DATA_DIR}/persian_typo_dataset_normalized"
dataset.save_to_disk(normalized_path)
print(f"\n💾 نسخه نرمال‌شده در Drive ذخیره شد: {normalized_path}")

# ---------- گام ۴: توکنایز ----------
print("\n🔧 گام ۳: توکنایز کردن...")
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["input"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length",
    )
    labels = tokenizer(
        examples["target"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length",
    )
    model_inputs["labels"] = labels["input_ids"]

    # جایگزینی pad با -100 تا در loss نادیده گرفته شود
    model_inputs["labels"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in model_inputs["labels"]
    ]
    return model_inputs

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

print("\n" + "="*60)
print("✅ توکنایز کامل شد")
print("="*60)
print(f"📊 Train: {len(tokenized_dataset['train'])}")
print(f"📊 Validation: {len(tokenized_dataset['validation'])}")
print(f"📊 ستون‌های توکنایز‌شده: {tokenized_dataset['train'].column_names}")
print(f"📊 شکل نمونه: {len(tokenized_dataset['train'][0]['input_ids'])} توکن")
print("="*60)

🔧 گام ۱: نرمال‌سازی نام ستون‌ها...
   ستون‌های فعلی: ['with_typos', 'fixed']
   ✅ with_typos → input, fixed → target

🔧 گام ۲: بررسی تقسیم train/validation...
   ⚠️ validation وجود ندارد → تقسیم ۹۰/۱۰ انجام می‌شود
   ✅ train: 20212
   ✅ validation: 2246


Saving the dataset (0/1 shards):   0%|          | 0/20212 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2246 [00:00<?, ? examples/s]


💾 نسخه نرمال‌شده در Drive ذخیره شد: /content/drive/MyDrive/pars_corrector/data/persian_typo_dataset_normalized

🔧 گام ۳: توکنایز کردن...


Map:   0%|          | 0/20212 [00:00<?, ? examples/s]

Map:   0%|          | 0/2246 [00:00<?, ? examples/s]


✅ توکنایز کامل شد
📊 Train: 20212
📊 Validation: 2246
📊 ستون‌های توکنایز‌شده: ['input_ids', 'attention_mask', 'labels']
📊 شکل نمونه: 128 توکن


# 🩺 بعد از اجرا، این سلول را بزن تا از سلامت داده مطمئن شوی


In [8]:
#@title 🩺 تست کیفیت داده
sample = dataset["train"][0]
print("📝 ورودی (با غلط):")
print(f"   {sample['input']}")
print("\n✅ خروجی (صحیح):")
print(f"   {sample['target']}")

# محاسبه درصد تفاوت کاراکتری
import difflib
similarity = difflib.SequenceMatcher(None, sample['input'], sample['target']).ratio()
print(f"\n📊 شباهت کاراکتری: {similarity*100:.1f}%")
print("   (بالای ۹۰٪ یعنی غلط‌ها ظریف‌اند — برای BLEU 0.9+ عالی است)")

# چند نمونه دیگر
print("\n" + "="*60)
print("📋 ۵ نمونه تصادفی دیگر:")
print("="*60)
import random
random.seed(42)
for i in random.sample(range(len(dataset['train'])), 5):
    s = dataset['train'][i]
    # پیدا کردن تفاوت
    diff = difflib.ndiff(s['input'].split(), s['target'].split())
    changes = [d for d in diff if d.startswith('- ') or d.startswith('+ ')]
    print(f"\n{i}. غلط: {s['input'][:80]}...")
    print(f"   صحیح: {s['target'][:80]}...")
    if changes[:4]:
        print(f"   تغییرات: {changes[:4]}")

📝 ورودی (با غلط):
   دانش آموزان و دانشجویان باید در فعالییتهای اجتماعی و فرهنگی شرکت کنن

✅ خروجی (صحیح):
   دانش آموزان و دانشجویان باید در فعالیت‌های اجتماعی و فرهنگی شرکت کنند

📊 شباهت کاراکتری: 97.8%
   (بالای ۹۰٪ یعنی غلط‌ها ظریف‌اند — برای BLEU 0.9+ عالی است)

📋 ۵ نمونه تصادفی دیگر:

3648. غلط: شرکت های تولیدی خودرو برای رفع مشکل کمبود قطعه از قطعات چینی استفاده میکنند....
   صحیح: شرکت‌های تولیدی خودرو برای رفع مشکل کمبود قطعه از قطعات چینی استفاده می‌کنند....
   تغییرات: ['+ شرکت\u200cهای', '- شرکت', '- های', '- میکنند.']

819. غلط: شرکت های داخلی در نمایشگاه بین المللی تهران اخرین محصولات خود را عرزه کردن....
   صحیح: شرکت‌های داخلی در نمایشگاه بین‌المللی تهران آخرین محصولات خود را عرضه کردند....
   تغییرات: ['+ شرکت\u200cهای', '- شرکت', '- های', '- بین']

9012. غلط: تیم ملی والیبال ایران در مقابل تیم روسیه به پیروزی رسید. تماشاگران از این برد خو...
   صحیح: تیم ملی والیبال ایران در مقابل تیم روسیه به پیروزی رسید. تماشاگران از این برد خو...
   تغییرات: ['+ خوشحال', '- خوش', '

# 🟦 سلول ۶ — تنظیمات آموزش


In [9]:
#@title 6. تنظیمات آموزش (سازگار با transformers 5.x)
from transformers import Seq2SeqTrainingArguments
import inspect

# تشخیص نام آرگومان ارزیابی (در نسخه‌های جدید: eval_strategy، قدیمی: evaluation_strategy)
sig = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
eval_arg_name = "eval_strategy" if "eval_strategy" in sig else "evaluation_strategy"
print(f"🔍 آرگومان ارزیابی در این نسخه: {eval_arg_name}")

training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,

    # ---------- آموزش ----------
    num_train_epochs=5,
    per_device_train_batch_size=8,          # T4: 8 امن
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,          # batch مؤثر = 16
    learning_rate=3e-4,
    warmup_steps=500,
    weight_decay=0.01,
    max_grad_norm=1.0,

    # ---------- ارزیابی و ذخیره ----------
    **{eval_arg_name: "steps"},
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    logging_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # ---------- بهینه‌سازی T4 ----------
    fp16=True,
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,

    # ---------- سایر ----------
    report_to="none",
    push_to_hub=False,
    seed=42,
    dataloader_num_workers=2,
)

eff_batch = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
steps_per_epoch = len(tokenized_dataset["train"]) // eff_batch
total_steps = steps_per_epoch * training_args.num_train_epochs

print("\n" + "="*60)
print("✅ تنظیمات آماده شد")
print("="*60)
print(f"📊 batch مؤثر: {eff_batch}")
print(f"📊 steps/epoch: {steps_per_epoch}")
print(f"📊 total steps: {total_steps}")
print(f"📊 تعداد ارزیابی‌ها: {total_steps // 500}")
print(f"📊 fp16: {training_args.fp16}")
print(f"📊 lr: {training_args.learning_rate}")
print(f"📊 epochs: {training_args.num_train_epochs}")
print("="*60)

🔍 آرگومان ارزیابی در این نسخه: eval_strategy

✅ تنظیمات آماده شد
📊 batch مؤثر: 16
📊 steps/epoch: 1263
📊 total steps: 6315
📊 تعداد ارزیابی‌ها: 12
📊 fp16: True
📊 lr: 0.0003
📊 epochs: 5


# 🟦 سلول ۷ — متریک BLEU


In [10]:
#@title 7. متریک BLEU (با نصب خودکار sacrebleu)
import subprocess, sys

try:
    import sacrebleu
    print("✅ sacrebleu موجود است")
except ImportError:
    print("⬇️ نصب sacrebleu ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sacrebleu"])
    print("✅ sacrebleu نصب شد")

import evaluate
import numpy as np

metric = evaluate.load("sacrebleu")
print("✅ متریک BLEU آماده است")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # در برخی نسخه‌ها predictions به‌صورت tuple برمی‌گردد
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # دیکد کردن پیش‌بینی‌ها
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # جایگزینی -100 با pad_token_id قبل از دیکد
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # محاسبه BLEU
    result = metric.compute(
        predictions=decoded_preds,
        references=[[l] for l in decoded_labels],
    )

    # BLEU + chrF برای ارزیابی کامل‌تر فارسی
    try:
        chrf = evaluate.load("chrf")
        chrf_result = chrf.compute(
            predictions=decoded_preds,
            references=[[l] for l in decoded_labels],
        )
        return {"bleu": result["score"], "chrf": chrf_result["score"]}
    except Exception:
        return {"bleu": result["score"]}

⬇️ نصب sacrebleu ...
✅ sacrebleu نصب شد


✅ متریک BLEU آماده است


# 🟦 سلول ۸ — ساخت Trainer


In [11]:
#@title 8. ساخت Trainer (سازگار با transformers 5.x)
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq
import inspect

# ---------- Data Collator ----------
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
)
print("✅ Data Collator ساخته شد")

# ---------- Trainer (تشخیص خودکار tokenizer vs processing_class) ----------
trainer_sig = inspect.signature(Seq2SeqTrainer.__init__).parameters

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

if "processing_class" in trainer_sig:
    trainer_kwargs["processing_class"] = tokenizer
    print("🔍 استفاده از processing_class (نسخه جدید)")
elif "tokenizer" in trainer_sig:
    trainer_kwargs["tokenizer"] = tokenizer
    print("🔍 استفاده از tokenizer (نسخه قدیمی)")

trainer = Seq2SeqTrainer(**trainer_kwargs)
print("✅ Trainer ساخته شد")

# ---------- خلاصه ----------
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print("\n" + "="*60)
print("📊 خلاصه Trainer")
print("="*60)
print(f"پارامترهای قابل آموزش: {trainable:,} / {total:,} ({trainable/total*100:.1f}%)")
print(f"دستگاه: {training_args.device}")
print(f"fp16: {training_args.fp16}")
print(f"mixed precision: {training_args.fp16}")
print("="*60)

✅ Data Collator ساخته شد
🔍 استفاده از processing_class (نسخه جدید)
✅ Trainer ساخته شد

📊 خلاصه Trainer
پارامترهای قابل آموزش: 300,176,768 / 300,176,768 (100.0%)
دستگاه: cuda:0
fp16: True
mixed precision: True


# 🟦 سلول ۹ — شروع آموزش (با قابلیت Resume خودکار)


In [ ]:
#@title 9. شروع آموزش (با Resume خودکار در صورت قطعی)
import glob, os

print("🚀 آماده‌سازی آموزش...")
print(f"📁 چک‌پوینت‌ها: {CHECKPOINT_DIR}")
print(f"📁 مدل نهایی: {MODEL_DIR}")

# ---------- بررسی چک‌پوینت موجود ----------
existing_ckpts = sorted(
    glob.glob(f"{CHECKPOINT_DIR}/checkpoint-*"),
    key=lambda p: int(p.split("-")[-1]),
)

resume_from = None
if existing_ckpts:
    resume_from = existing_ckpts[-1]
    print(f"\n🔄 چک‌پوینت موجود پیدا شد: {resume_from}")
    print("   آموزش از همان‌جا ادامه می‌یابد.")
else:
    print("\n🆕 چک‌پوینتی وجود ندارد → آموزش از صفر شروع می‌شود.")

# ---------- تخمین زمان ----------
total_steps = int(
    training_args.num_train_epochs
    * len(tokenized_dataset["train"])
    // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
)
print(f"\n⏱️ زمان تقریبی روی T4: {total_steps * 1.5 / 60:.0f} تا {total_steps * 2.5 / 60:.0f} دقیقه")
print(f"📊 تعداد کل steps: {total_steps}")
print(f"📊 ارزیابی هر {training_args.eval_steps} step")
print(f"📊 ذخیره هر {training_args.save_steps} step")
print("\n" + "="*60)
print("⏳ شروع آموزش... (تب مرورگر را باز نگه دار)")
print("="*60 + "\n")

# ---------- آموزش ----------
train_result = trainer.train(resume_from_checkpoint=resume_from)

print("\n" + "="*60)
print("✅ آموزش کامل شد!")
print("="*60)
print(f"📊 Loss نهایی آموزش: {train_result.training_loss:.4f}")
print(f"📊 تعداد کل steps: {train_result.global_step}")
print(f"📊 زمان کل: {train_result.metrics.get('train_runtime', 0)/60:.1f} دقیقه")
print("="*60)

# ✅ کد کامل سلول ۱۰ (سازگار با transformers 5.21.0)


In [13]:
#@title 10. ارزیابی نهایی و ذخیره مدل در Drive
import os
import json
from datetime import datetime

print("="*60)
print("📊 ارزیابی نهایی روی validation set")
print("="*60)
print("⏳ در حال محاسبه BLEU و chrF... (۱–۳ دقیقه)\n")

# ---------- ارزیابی ----------
eval_results = trainer.evaluate()

# ---------- نمایش نتایج ----------
print("\n" + "="*60)
print("📊 نتایج ارزیابی نهایی")
print("="*60)
for k, v in eval_results.items():
    if isinstance(v, float):
        print(f"  {k:25s}: {v:.4f}")
    else:
        print(f"  {k:25s}: {v}")
print("="*60)

# ---------- ذخیره مدل نهایی در Drive ----------
FINAL_MODEL_PATH = f"{MODEL_DIR}/pars_corrector_mt5_small_v1"
os.makedirs(FINAL_MODEL_PATH, exist_ok=True)

print(f"\n💾 ذخیره مدل در Drive: {FINAL_MODEL_PATH}")

# مدل و توکنایزر
trainer.save_model(FINAL_MODEL_PATH)
tokenizer.save_pretrained(FINAL_MODEL_PATH)

# خاموش‌کردن هشدار tie و ذخیره‌ی config تمیز
model.config.tie_word_embeddings = False
model.config.save_pretrained(FINAL_MODEL_PATH)

# ---------- ذخیره‌ی متادیتای پروژه ----------
metadata = {
    "project_name": "pars_corrector",
    "model_name": MODEL_NAME,
    "created_at": datetime.now().isoformat(),
    "num_train_samples": len(tokenized_dataset["train"]),
    "num_validation_samples": len(tokenized_dataset["validation"]),
    "num_epochs": training_args.num_train_epochs,
    "batch_size": training_args.per_device_train_batch_size,
    "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
    "learning_rate": training_args.learning_rate,
    "fp16": training_args.fp16,
    "eval_results": {k: (float(v) if isinstance(v, (int, float)) else str(v))
                     for k, v in eval_results.items()},
    "total_params": sum(p.numel() for p in model.parameters()),
    "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
    "max_input_length": MAX_INPUT_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
}

metadata_path = f"{FINAL_MODEL_PATH}/project_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(f"📋 متادیتا ذخیره شد: project_metadata.json")

# ---------- نمایش فایل‌های ذخیره‌شده ----------
print("\n📁 فایل‌های ذخیره‌شده:")
total_size = 0
for f in sorted(os.listdir(FINAL_MODEL_PATH)):
    fpath = os.path.join(FINAL_MODEL_PATH, f)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        total_size += size_mb
        print(f"  - {f:40s} ({size_mb:7.2f} MB)")

print(f"\n📦 حجم کل: {total_size:.1f} MB")
print(f"📍 مسیر کامل: {FINAL_MODEL_PATH}")

# ---------- بررسی هدف BLEU ----------
bleu = eval_results.get("eval_bleu", 0)
chrf = eval_results.get("eval_chrf", 0)
eval_loss = eval_results.get("eval_loss", float("inf"))

print("\n" + "="*60)
print("🎯 ارزیابی نهایی")
print("="*60)
print(f"BLEU: {bleu:.2f}  |  chrF: {chrf:.2f}  |  Loss: {eval_loss:.4f}")

if bleu >= 90:
    status = "🎯 عالی! هدف BLEU 0.9+ محقق شد."
elif bleu >= 85:
    status = "✅ خوب! نزدیک به هدف."
    status += "\n   💡 می‌توانی ۲ epoch دیگر آموزش بدهی."
elif bleu >= 75:
    status = "⚠️ متوسط. برای بهبود:"
    status += "\n   - ۳ epoch دیگر آموزش بده"
    status += "\n   - learning_rate را کمی کمتر کن (2e-4)"
else:
    status = "❌ پایین. پیشنهاد:"
    status += "\n   - بررسی کن که دیتاست با کیفیت باشد"
    status += "\n   - ۵ epoch دیگر آموزش بده"
    status += "\n   - از مدل پایه بزرگ‌تر (mt5-base) استفاده کن"

print(f"\n{status}")

# ---------- بررسی سلامت مدل ----------
print("\n" + "="*60)
print("🩺 بررسی نهایی")
print("="*60)

# ۱. فایل مدل
model_file = None
for fname in ["model.safetensors", "pytorch_model.bin"]:
    fpath = os.path.join(FINAL_MODEL_PATH, fname)
    if os.path.exists(fpath):
        model_file = fname
        break

if model_file:
    print(f"✅ فایل مدل: {model_file}")
else:
    print("❌ فایل مدل پیدا نشد! سلول را دوباره اجرا کن.")

# ۲. توکنایزر
tok_files = ["spiece.model", "tokenizer_config.json", "special_tokens_map.json"]
missing = [f for f in tok_files if not os.path.exists(os.path.join(FINAL_MODEL_PATH, f))]
if not missing:
    print(f"✅ فایل‌های توکنایزر کامل")
else:
    print(f"⚠️ فایل‌های توکنایزر ناقص: {missing}")

# ۳. config
if os.path.exists(os.path.join(FINAL_MODEL_PATH, "config.json")):
    print(f"✅ config.json موجود")
else:
    print("❌ config.json وجود ندارد!")

# ۴. هندل کردن NaN
if eval_loss != eval_loss:  # NaN check
    print("❌ Loss نهایی NaN است! آموزش مشکل داشته.")

print("\n" + "="*60)
print("✅ سلول ۱۰ کامل شد.")
print("📌 قدم بعدی: سلول ۱۱ (تست مدل با مثال‌های واقعی)")
print("="*60)

📊 ارزیابی نهایی روی validation set
⏳ در حال محاسبه BLEU و chrF... (۱–۳ دقیقه)



Training Loss,Validation Loss,Step,Bleu,Chrf
0.000000,nan,6320,0.000000,0.001661



📊 نتایج ارزیابی نهایی
  eval_loss                : nan
  eval_bleu                : 0.0000
  eval_chrf                : 0.0017

💾 ذخیره مدل در Drive: /content/drive/MyDrive/pars_corrector/models/pars_corrector_mt5_small_v1


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

📋 متادیتا ذخیره شد: project_metadata.json

📁 فایل‌های ذخیره‌شده:
  - config.json                              (   0.00 MB)
  - generation_config.json                   (   0.00 MB)
  - model.safetensors                        (1200.73 MB)
  - project_metadata.json                    (   0.00 MB)
  - tokenizer.json                           (  16.35 MB)
  - tokenizer_config.json                    (   0.00 MB)
  - training_args.bin                        (   0.01 MB)

📦 حجم کل: 1217.1 MB
📍 مسیر کامل: /content/drive/MyDrive/pars_corrector/models/pars_corrector_mt5_small_v1

🎯 ارزیابی نهایی
BLEU: 0.00  |  chrF: 0.00  |  Loss: nan

❌ پایین. پیشنهاد:
   - بررسی کن که دیتاست با کیفیت باشد
   - ۵ epoch دیگر آموزش بده
   - از مدل پایه بزرگ‌تر (mt5-base) استفاده کن

🩺 بررسی نهایی
✅ فایل مدل: model.safetensors
⚠️ فایل‌های توکنایزر ناقص: ['spiece.model', 'special_tokens_map.json']
✅ config.json موجود
❌ Loss نهایی NaN است! آموزش مشکل داشته.

✅ سلول ۱۰ کامل شد.
📌 قدم بعدی: سلول ۱۱ (تست مدل با مثال‌